## LEVEL 3
1. Parse JSON fields where applicable
2. Add priceLow, priceHigh, areacode, lat, lon
3. Drop fields where applicable

#### Initialize

In [2]:
import sys
from pathlib import Path
import pandas as pd

HERE = Path.cwd()
PARENT = HERE.parent.parent.parent  # server/scripts
if str(PARENT) not in sys.path:
    sys.path.insert(0, str(PARENT))
LEVEL2_BUCKET_PATH = PARENT / "server/out/places_level2"
LEVEL3_BUCKET_PATH = PARENT / "server/out/places_level3"
LEVEL3_BUCKET_PATH.mkdir(parents=True, exist_ok=True)
LEVEL2_BUCKET = [f for f in LEVEL2_BUCKET_PATH.rglob("*.csv") if f.is_file() and f.stem != "matched"]
DF_LEVEL2 = pd.concat([pd.read_csv(f) for f in LEVEL2_BUCKET], ignore_index=True)

Check Validity

In [3]:
df_validity = pd.DataFrame({ 
    col: DF_LEVEL2[col].notna().sum() / len(DF_LEVEL2) 
    for col in DF_LEVEL2.columns 
}, index=[0])
display(df_validity)

,id,displayName,primaryTypeDisplayName,rating,userRatingCount,location,shortFormattedAddress,googleMapsUri,priceRange,priceLevel,...,tile_path_id,seed_index,level,chain_name,is_major_chain,is_chain,predictedType,chain_count,cuisineType,venueType
0,1.0,1.0,0.999178,1.0,1.0,1.0,1.0,1.0,0.823939,0.476149,...,1.0,1.0,1.0,0.238513,1.0,1.0,0.71351,1.0,1.0,1.0


#### Parse JSON

In [4]:
import ast
from server.scripts.clean_places_level_3.wheelchair_level import wheelchair_level
df_level3 = DF_LEVEL2.copy()
df_level3["location"] = df_level3["location"].apply(ast.literal_eval)
df_level3["lat"] = df_level3["location"].apply(lambda x: x.get("latitude") if isinstance(x, dict) else None)
df_level3["lon"] = df_level3["location"].apply(lambda x: x.get("longitude") if isinstance(x, dict) else None)
df_level3.drop(columns=["location"], inplace=True)

valid_postaladdress = df_level3.loc[df_level3["postalAddress"].notna(), "postalAddress"]
df_level3["postalAddress"] = valid_postaladdress.apply(ast.literal_eval)
df_level3["pcd"] = df_level3["postalAddress"].apply(lambda x: x.get("postalCode") if isinstance(x, dict) else "")
df_level3["areacode"] = df_level3["pcd"].apply(lambda x: x.split(" ")[0] if isinstance(x, str) and " " in x else "")
df_level3.drop(columns=["postalAddress"], inplace=True)

valid_pricerange = df_level3.loc[df_level3["priceRange"].notna(), "priceRange"]
df_level3["priceRange"] = valid_pricerange.apply(ast.literal_eval)
df_level3["startPrice"] = df_level3['priceRange'].apply(lambda x: x.get("startPrice", {}).get("units") if isinstance(x, dict) else None).astype(float)
df_level3["endPrice"] = df_level3['priceRange'].apply(lambda x: x.get("endPrice", {}).get("units") if isinstance(x, dict) else None).astype(float)
df_level3.drop(columns=["priceRange"], inplace=True)

valid_accessibility = df_level3.loc[df_level3["accessibilityOptions"].notna(), "accessibilityOptions"]
df_level3["accessibilityOptions"] = valid_accessibility.apply(ast.literal_eval)
df_level3["wheelchairAccess"] = df_level3["accessibilityOptions"].apply(lambda x: wheelchair_level(x))
df_level3.drop(columns=["accessibilityOptions"], inplace=True)

df_level3['operational'] = df_level3['businessStatus'] == 'OPERATIONAL'
df_level3.drop(columns=["businessStatus"], inplace=True)

df_level3.drop(
    columns=["addressComponents", "addressDescriptor", "regularOpeningHours", "containingPlaces"], 
    inplace=True
)

Parse Price

In [5]:
from server.scripts.clean_places_level_3.cost_category import categorize_cost

df_level3['cost'] = df_level3.apply(lambda row: categorize_cost(row['startPrice'], row['endPrice'], row['priceLevel']), axis=1)
df_level3.drop(columns=["startPrice", "endPrice", "priceLevel"], inplace=True)
df_level3[['displayName', 'cost']]
# sample = df_level3[~df_level3['priceLevel'].notna() & ~df_level3['startPrice'].notna() & ~df_level3['endPrice'].notna()]
# sample[["startPrice", "endPrice", "priceLevel"]]

,displayName,cost
0,"Tossed, London Wall",10+
1,Honest Burgers Bank,10+
2,McDonald's,<10
3,Acer Restaurant,10+
4,K10 Bank,<10
...,...,...
18233,Mama Nati Ltd,NaN
18234,Everyman Crystal Palace,NaN
18235,Curzon Aldgate,NaN
18236,FC Soper Fishmonger,NaN


#### Drop Fields

In [6]:
# ROW1 = df_level3.iloc[0]
# display(ROW1["accessibilityOptions"])
df_level3_2 = df_level3.copy()
df_level3_2.drop(
    columns=['tile_id', 'tile_path_id', 'seed_index', 'level', 'pureServiceAreaBusiness'], 
    inplace=True
)
display(df_level3.columns)

Index(['id', 'displayName', 'primaryTypeDisplayName', 'rating',
       'userRatingCount', 'shortFormattedAddress', 'googleMapsUri',
       'websiteUri', 'types', 'primaryType', 'pureServiceAreaBusiness',
       'tile_id', 'tile_path_id', 'seed_index', 'level', 'chain_name',
       'is_major_chain', 'is_chain', 'predictedType', 'chain_count',
       'cuisineType', 'venueType', 'lat', 'lon', 'pcd', 'areacode',
       'wheelchairAccess', 'operational', 'cost'],
      dtype='str')

#### Export

In [7]:
df_level3_2.to_csv(LEVEL3_BUCKET_PATH / "places_level3.csv", index=False)